In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

fatal: destination path 'CTAB-GAN-Plus' already exists and is not an empty directory.


In [2]:
from ucimlrepo import fetch_ucirepo

In [3]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sdv.metadata import SingleTableMetadata
import torch
import random

wine_quality = fetch_ucirepo(id=186)

X = wine_quality.data.features
y = wine_quality.data.targets

data = pd.concat([X, y], axis=1)

target_col = "quality"

data = data.replace("?", np.nan)
n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

numeric_cols = data.columns

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

data[target_col] = data[target_col].astype(int)

processed_data = data.copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


In [4]:
from sklearn.model_selection import train_test_split
from sdv.evaluation.single_table import evaluate_quality

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
    stratify=processed_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

try:
    data_path = "wine_quality_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=[target_col],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Classification": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    # Convert all columns back to numeric where possible
    for col in synthetic_ctabgan.columns:
        synthetic_ctabgan[col] = pd.to_numeric(
            synthetic_ctabgan[col],
            errors="coerce"
        )

        synthetic_ctabgan[col] = synthetic_ctabgan[col].fillna(
            train_real[col].median()
        )

    # Wine quality target is an integer score between 0 and 10
    synthetic_ctabgan[target_col] = (
        synthetic_ctabgan[target_col]
        .round()
        .clip(0, 10)
        .astype(int)
    )

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)


================ SINGLE RUN ================


100%|██████████| 150/150 [15:15<00:00,  6.11s/it]


Finished training in 923.1501333713531  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 211.28it/s]|
Column Shapes Score: 92.93%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 74.35it/s]|
Column Pair Trends Score: 90.69%

Overall Score (Average): 91.81%

CTABGAN: 0.9181


In [12]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    encoder = LabelEncoder()
    data_wgan[target_col] = encoder.fit_transform(data_wgan[target_col])

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    max_label_code = len(encoder.classes_) - 1

    synthetic_wgan[target_col] = (
        synthetic_wgan[target_col]
        .round()
        .clip(0, max_label_code)
        .astype(int)
    )

    synthetic_wgan[target_col] = encoder.inverse_transform(
        synthetic_wgan[target_col]
    )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 271.71it/s]|
Column Shapes Score: 87.17%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 297.64it/s]|
Column Pair Trends Score: 88.61%

Overall Score (Average): 87.89%

WGAN_GP: 0.8789


In [6]:
# SDV MODELS

from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_data[target_col] = (
            synthetic_data[target_col]
            .round()
            .clip(0, 10)
            .astype(int)
        )

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 112.16it/s]|
Column Shapes Score: 74.56%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 84.02it/s]|
Column Pair Trends Score: 83.42%

Overall Score (Average): 78.99%

CTGAN: 0.7899
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 151.51it/s]|
Column Shapes Score: 75.13%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 152.32it/s]|
Column Pair Trends Score: 82.6%

Overall Score (Average): 78.86%

CopulaGAN: 0.7886
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 342.77it/s]|
Column Shapes Score: 89.31%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 221.65it/s]|
Column Pair Trends Score: 89.64%

Overall Score (Average): 89.48%

TVAE: 0.8948
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 353.48it/s]|
Column Shapes Sc

In [7]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=42),
    'Lasso': Lasso(random_state=42),
    'ElasticNet': ElasticNet(random_state=42),
    'SVR-RBF': SVR(kernel='rbf'),
    'KNN': KNeighborsRegressor(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'ExtraTrees': ExtraTreesRegressor(random_state=42),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
    'MLP': MLPRegressor(max_iter=2000, random_state=42)
}


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None
):
    if seeds is None:
        seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results = []

    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in train_df.columns:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        train_df[col] = train_df[col].fillna(train_df[col].median())

    for col in test_df.columns:
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")
        test_df[col] = test_df[col].fillna(train_df[col].median())

    train_df[label_col] = train_df[label_col].round().clip(0, 10).astype(int)
    test_df[label_col] = test_df[label_col].round().clip(0, 10).astype(int)

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]

            X_test_full = test_df.drop(columns=[label_col])
            y_test_full = test_df[label_col]

            stratify_train = (
                y_train_full
                if y_train_full.nunique() > 1 and y_train_full.value_counts().min() >= 2
                else None
            )

            X_train_split, _, y_train_split, _ = train_test_split(
                X_train_full,
                y_train_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_train
            )

            stratify_test = (
                y_test_full
                if y_test_full.nunique() > 1 and y_test_full.value_counts().min() >= 2
                else None
            )

            _, X_test_split, _, y_test_split = train_test_split(
                X_test_full,
                y_test_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_test
            )

            if y_train_split.nunique() < 2:
                continue

            scaler = StandardScaler()

            X_train_s = scaler.fit_transform(X_train_split)
            X_test_s = scaler.transform(X_test_split)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            try:
                clf.fit(X_train_s, y_train_split)

                y_pred = clf.predict(X_test_s)

                accuracy_scores.append(accuracy_score(y_test_split, y_pred))

                f1_scores.append(
                    f1_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                precision_scores.append(
                    precision_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                recall_scores.append(
                    recall_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

            except Exception:
                continue

        if len(accuracy_scores) == 0:
            results.append({
                "Model": name,
                "Accuracy Mean": np.nan,
                "Accuracy Std": np.nan,
                "F1 Mean": np.nan,
                "F1 Std": np.nan,
                "Precision Mean": np.nan,
                "Precision Std": np.nan,
                "Recall Mean": np.nan,
                "Recall Std": np.nan,
                "Accuracy ± SD": "N/A",
                "F1 ± SD": "N/A",
                "Precision ± SD": "N/A",
                "Recall ± SD": "N/A"
            })
            continue

        acc_mean = np.mean(accuracy_scores)
        acc_std = np.std(accuracy_scores, ddof=1) if len(accuracy_scores) > 1 else 0

        f1_mean = np.mean(f1_scores)
        f1_std = np.std(f1_scores, ddof=1) if len(f1_scores) > 1 else 0

        prec_mean = np.mean(precision_scores)
        prec_std = np.std(precision_scores, ddof=1) if len(precision_scores) > 1 else 0

        rec_mean = np.mean(recall_scores)
        rec_std = np.std(recall_scores, ddof=1) if len(recall_scores) > 1 else 0

        results.append({
            "Model": name,
            "Accuracy Mean": acc_mean,
            "Accuracy Std": acc_std,
            "F1 Mean": f1_mean,
            "F1 Std": f1_std,
            "Precision Mean": prec_mean,
            "Precision Std": prec_std,
            "Recall Mean": rec_mean,
            "Recall Std": rec_std,
            "Accuracy ± SD": f"{acc_mean:.4f} ± {acc_std:.4f}",
            "F1 ± SD": f"{f1_mean:.4f} ± {f1_std:.4f}",
            "Precision ± SD": f"{prec_mean:.4f} ± {prec_std:.4f}",
            "Recall ± SD": f"{rec_mean:.4f} ± {rec_std:.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False,
        na_position="last"
    )

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

X = processed_data.drop(columns=[target_col])
y = processed_data[target_col]

print("Target column:", target_col)
print("Target classes:")
print(y.value_counts())

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        scaler = StandardScaler()

        X_train_real = scaler.fit_transform(X_train_real)
        X_test_real = scaler.transform(X_test_real)

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores, ddof=1)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores, ddof=1)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores, ddof=1)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores, ddof=1)

    trtr_results.append({
        "Model": model_name,
        "Accuracy Mean_TRTR": acc_mean,
        "Accuracy Std_TRTR": acc_std,
        "F1 Mean_TRTR": f1_mean,
        "F1 Std_TRTR": f1_std,
        "Precision Mean_TRTR": prec_mean,
        "Precision Std_TRTR": prec_std,
        "Recall Mean_TRTR": rec_mean,
        "Recall Std_TRTR": rec_std,
        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results).sort_values(
    by="Accuracy Mean_TRTR",
    ascending=False
)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)

--- Starting TRTR Evaluation (Train Real, Test Real) ---
Target column: quality
Target classes:
quality
6    458
5    302
7    167
4     36
8     32
3      5
Name: count, dtype: int64
Running LogReg...
Running SVM-RBF...
Running KNN...
Running NaiveBayes...
Running DecisionTree...
Running RandomForest...
Running ExtraTrees...
Running GradientBoost...
Running AdaBoost...
Running MLP...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
5,RandomForest,0.5350 ± 0.0224,0.5096 ± 0.0204,0.5085 ± 0.0171,0.5350 ± 0.0224
6,ExtraTrees,0.5325 ± 0.0240,0.5076 ± 0.0229,0.5093 ± 0.0264,0.5325 ± 0.0240
7,GradientBoost,0.5285 ± 0.0293,0.5114 ± 0.0297,0.5115 ± 0.0396,0.5285 ± 0.0293
1,SVM-RBF,0.5255 ± 0.0331,0.4765 ± 0.0340,0.5045 ± 0.0426,0.5255 ± 0.0331
0,LogReg,0.5120 ± 0.0289,0.4689 ± 0.0269,0.4875 ± 0.0468,0.5120 ± 0.0289
2,KNN,0.4905 ± 0.0266,0.4727 ± 0.0254,0.4749 ± 0.0350,0.4905 ± 0.0266
9,MLP,0.4745 ± 0.0253,0.4716 ± 0.0258,0.4726 ± 0.0276,0.4745 ± 0.0253
8,AdaBoost,0.4595 ± 0.0379,0.4241 ± 0.0316,0.4244 ± 0.0357,0.4595 ± 0.0379
4,DecisionTree,0.4460 ± 0.0503,0.4469 ± 0.0484,0.4514 ± 0.0476,0.4460 ± 0.0503
3,NaiveBayes,0.4115 ± 0.0513,0.4141 ± 0.0505,0.4373 ± 0.0505,0.4115 ± 0.0513


In [13]:
import pandas as pd
import numpy as np

label_col = target_col

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

real_data = processed_data.copy()

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=real_data,
    test_df=real_data,
    label="quality",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy ± SD",
            "F1 ± SD",
            "Precision ± SD",
            "Recall ± SD"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"{synth_name} not found in synthetic_datasets. Skipping.")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name].copy()

    if label_col not in synthetic_train_df.columns:
        print(f"{label_col} not found in {synth_name}. Skipping.")
        continue

    synthetic_train_df[label_col] = pd.to_numeric(
        synthetic_train_df[label_col],
        errors="coerce"
    )

    synthetic_train_df[label_col] = (
        synthetic_train_df[label_col]
        .fillna(real_data[label_col].mode()[0])
        .round()
        .astype(int)
    )

    synthetic_train_df = synthetic_train_df.dropna()

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=real_data,
        label="quality",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy ± SD",
                "F1 ± SD",
                "Precision ± SD",
                "Recall ± SD"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy ± SD_TRTR",
                "Accuracy ± SD_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby(
        "Synthetic_Model",
        as_index=False
    )[
        [
            "Accuracy_Drop",
            "F1_Drop",
            "Precision_Drop",
            "Recall_Drop"
        ]
    ]
    .mean()
    .sort_values(
        "Accuracy_Drop",
        ascending=True
    )
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
5,RandomForest,0.5350 ± 0.0224,0.5096 ± 0.0204,0.5085 ± 0.0171,0.5350 ± 0.0224
6,ExtraTrees,0.5325 ± 0.0240,0.5076 ± 0.0229,0.5093 ± 0.0264,0.5325 ± 0.0240
7,GradientBoost,0.5285 ± 0.0293,0.5114 ± 0.0297,0.5115 ± 0.0396,0.5285 ± 0.0293
1,SVM-RBF,0.5255 ± 0.0331,0.4765 ± 0.0340,0.5045 ± 0.0426,0.5255 ± 0.0331
0,LogReg,0.5120 ± 0.0289,0.4689 ± 0.0269,0.4875 ± 0.0468,0.5120 ± 0.0289
2,KNN,0.4905 ± 0.0266,0.4727 ± 0.0254,0.4749 ± 0.0350,0.4905 ± 0.0266
9,MLP,0.4745 ± 0.0253,0.4716 ± 0.0258,0.4726 ± 0.0276,0.4745 ± 0.0253
8,AdaBoost,0.4595 ± 0.0379,0.4241 ± 0.0316,0.4244 ± 0.0357,0.4595 ± 0.0379
4,DecisionTree,0.4460 ± 0.0503,0.4469 ± 0.0484,0.4514 ± 0.0476,0.4460 ± 0.0503
3,NaiveBayes,0.4115 ± 0.0513,0.4141 ± 0.0505,0.4373 ± 0.0505,0.4115 ± 0.0513


CTGAN - TSTR


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
0,LogReg,0.3220 ± 0.0359,0.3209 ± 0.0390,0.3450 ± 0.0365,0.3220 ± 0.0359
8,AdaBoost,0.2920 ± 0.0313,0.2593 ± 0.0347,0.3404 ± 0.0517,0.2920 ± 0.0313
3,NaiveBayes,0.2715 ± 0.0287,0.2596 ± 0.0326,0.3263 ± 0.0354,0.2715 ± 0.0287
5,RandomForest,0.2675 ± 0.0284,0.2560 ± 0.0308,0.3261 ± 0.0404,0.2675 ± 0.0284
6,ExtraTrees,0.2645 ± 0.0251,0.2563 ± 0.0273,0.3226 ± 0.0355,0.2645 ± 0.0251
1,SVM-RBF,0.2615 ± 0.0284,0.2505 ± 0.0320,0.3138 ± 0.0498,0.2615 ± 0.0284
7,GradientBoost,0.2380 ± 0.0183,0.2469 ± 0.0252,0.3282 ± 0.0300,0.2380 ± 0.0183
4,DecisionTree,0.2235 ± 0.0280,0.2472 ± 0.0310,0.3399 ± 0.0434,0.2235 ± 0.0280
2,KNN,0.2165 ± 0.0211,0.2193 ± 0.0210,0.3206 ± 0.0365,0.2165 ± 0.0211
9,MLP,0.1745 ± 0.0293,0.1917 ± 0.0318,0.2894 ± 0.0608,0.1745 ± 0.0293


CTGAN - TRTR vs TSTR


...

,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
0,LogReg,0.4620 ± 0.0059,0.3011 ± 0.0124,0.3139 ± 0.1057,0.4620 ± 0.0059
1,SVM-RBF,0.4600 ± 0.0075,0.3025 ± 0.0156,0.3214 ± 0.0921,0.4600 ± 0.0075
6,ExtraTrees,0.4340 ± 0.0246,0.3322 ± 0.0258,0.3348 ± 0.0520,0.4340 ± 0.0246
5,RandomForest,0.4235 ± 0.0283,0.3366 ± 0.0305,0.3417 ± 0.0559,0.4235 ± 0.0283
8,AdaBoost,0.4190 ± 0.0385,0.3476 ± 0.0326,0.3326 ± 0.0426,0.4190 ± 0.0385
3,NaiveBayes,0.4080 ± 0.0390,0.3512 ± 0.0325,0.3400 ± 0.0405,0.4080 ± 0.0390
7,GradientBoost,0.4075 ± 0.0318,0.3639 ± 0.0329,0.3583 ± 0.0383,0.4075 ± 0.0318
2,KNN,0.4015 ± 0.0267,0.3636 ± 0.0245,0.3449 ± 0.0296,0.4015 ± 0.0267
9,MLP,0.3480 ± 0.0287,0.3416 ± 0.0277,0.3387 ± 0.0272,0.3480 ± 0.0287
4,DecisionTree,0.3215 ± 0.0254,0.3243 ± 0.0258,0.3318 ± 0.0295,0.3215 ± 0.0254


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy ± SD_TRTR,Accuracy ± SD_TSTR
0,CTABGAN,RandomForest,0.1115,0.172956,0.166834,0.1115,0.5350 ± 0.0224,0.4235 ± 0.0283
1,CTABGAN,ExtraTrees,0.0985,0.175398,0.174506,0.0985,0.5325 ± 0.0240,0.4340 ± 0.0246
2,CTABGAN,GradientBoost,0.1210,0.147581,0.153255,0.1210,0.5285 ± 0.0293,0.4075 ± 0.0318
3,CTABGAN,SVM-RBF,0.0655,0.173988,0.183106,0.0655,0.5255 ± 0.0331,0.4600 ± 0.0075
4,CTABGAN,LogReg,0.0500,0.167818,0.173569,0.0500,0.5120 ± 0.0289,0.4620 ± 0.0059
5,CTABGAN,KNN,0.0890,0.109166,0.130040,0.0890,0.4905 ± 0.0266,0.4015 ± 0.0267
6,CTABGAN,MLP,0.1265,0.129934,0.133908,0.1265,0.4745 ± 0.0253,0.3480 ± 0.0287
7,CTABGAN,AdaBoost,0.0405,0.076543,0.091810,0.0405,0.4595 ± 0.0379,0.4190 ± 0.0385
8,CTABGAN,DecisionTree,0.1245,0.122625,0.119543,0.1245,0.4460 ± 0.0503,0.3215 ± 0.0254
9,CTABGAN,NaiveBayes,0.0035,0.062869,0.097281,0.0035,0.4115 ± 0.0513,0.4080 ± 0.0390


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
4,TVAE,0.00115,0.036342,0.022494,0.00115
5,WGAN_GP,0.06805,0.054127,0.054214,0.06805
0,CTABGAN,0.08305,0.133888,0.142385,0.08305
3,GaussianCopula,0.08940,0.133337,0.152659,0.08940
2,CopulaGAN,0.19960,0.171771,0.148419,0.19960
1,CTGAN,0.23840,0.219568,0.152944,0.23840


In [14]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
